# 05 — Construcción del endpoint experimental y datasets de modelado

**TFM:** Evaluación del valor incremental de parámetros ECG en riesgo cardiometabólico  
**Proceso:** construcción de `RIESGO_CARDIOMETABOLICO` y generación de escenarios experimentales E1–E4.

Este notebook toma como entrada la cohorte multimodal integrada y genera los datasets analíticos requeridos para el entrenamiento y evaluación de modelos predictivos. La variable objetivo se construye como un endpoint operacional académico basado en acumulación de factores de riesgo cardiometabólico. No constituye diagnóstico clínico ni escala clínica validada externamente.

## 1. Entradas y salidas

### Entrada principal

```text
Dataset_Multimodal_Integrado_TFM.xlsx
```

El archivo debe provenir de `04_integracion_ecg.ipynb` y contener la cohorte clínica integrada con variables NLP y parámetros ECG.

### Salidas principales

```text
Dataset_Endpoint_TFM.xlsx
Dataset_E1_Clinico_TFM.xlsx
Dataset_E2_Clinico_NLP_TFM.xlsx
Dataset_E3_Clinico_ECG_TFM.xlsx
Dataset_E4_Clinico_NLP_ECG_TFM.xlsx
Reporte_Endpoint_TFM.txt
```

Los datasets E1–E4 se utilizan como entrada de `06_modelado_predictivo.ipynb`.

In [1]:
# Instalación de dependencias requeridas
# Ejecutar esta celda si el entorno no tiene las dependencias instaladas.

import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
    else:
        print(f"Dependencia disponible: {modulo}")

Dependencia disponible: pandas
Dependencia disponible: numpy
Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter


In [2]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

## 2. Configuración de rutas

Ajustar `DATA_DIR` si los archivos se encuentran en otra carpeta. Por defecto se utiliza el directorio actual del notebook.

In [ ]:
DATA_DIR = Path(".").resolve()

INPUT_MULTIMODAL = DATA_DIR / "Dataset_Multimodal_Integrado_TFM.xlsx"

OUTPUT_ENDPOINT = DATA_DIR / "Dataset_Endpoint_TFM.xlsx"
OUTPUT_E1 = DATA_DIR / "Dataset_E1_Clinico_TFM.xlsx"
OUTPUT_E2 = DATA_DIR / "Dataset_E2_Clinico_NLP_TFM.xlsx"
OUTPUT_E3 = DATA_DIR / "Dataset_E3_Clinico_ECG_TFM.xlsx"
OUTPUT_E4 = DATA_DIR / "Dataset_E4_Clinico_NLP_ECG_TFM.xlsx"
OUTPUT_REPORT = DATA_DIR / "Reporte_Endpoint_TFM.txt"

RANDOM_STATE = 42
UMBRAL_RIESGO = 3

print("Directorio de trabajo:", DATA_DIR)
print("Entrada esperada:", INPUT_MULTIMODAL)

## 3. Funciones auxiliares

In [4]:
def read_excel_prefer_sheet(path: Path, preferred_sheet: str = "BASE_COMPLETA") -> pd.DataFrame:
    """Carga un Excel utilizando una hoja preferente si existe."""
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo requerido: {path}")
    xls = pd.ExcelFile(path)
    sheet = preferred_sheet if preferred_sheet in xls.sheet_names else xls.sheet_names[0]
    print(f"Archivo cargado: {path.name}")
    print(f"Hoja utilizada: {sheet}")
    return pd.read_excel(path, sheet_name=sheet)


def ensure_columns(df: pd.DataFrame, cols: list[str], context: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Columnas faltantes en {context}: {missing}")


def existing_columns(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]


BATERIA_COLUMN_CANDIDATES = [
    "SUBSET_BATERIA",
    "SUBSET_EXAMENES",
    "BATERIA",
    "BATERIA_ESTRUCTURAL",
    "BATERIA_CLUSTER",
    "CLUSTER_BATERIA",
    "subset_bateria",
    "bateria",
    "subset",
]


def normalize_bateria_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Estandariza la trazabilidad estructural de batería sin eliminar columnas originales."""
    df = df.copy()
    source_col = None
    for candidate in BATERIA_COLUMN_CANDIDATES:
        if candidate in df.columns:
            source_col = candidate
            break

    if source_col is None:
        warnings.warn("No se encontró columna de batería/subset. Se creará SUBSET_BATERIA = 'SIN_SUBSET'.")
        df["SUBSET_BATERIA"] = "SIN_SUBSET"
        source_col = "SUBSET_BATERIA"
    elif "SUBSET_BATERIA" not in df.columns:
        df["SUBSET_BATERIA"] = df[source_col]
    else:
        # SUBSET_BATERIA existe; se preserva como columna estándar transversal.
        source_col = "SUBSET_BATERIA"

    if "SUBSET_EXAMENES" in df.columns:
        # Se conserva la columna original del proceso de segmentación estructural.
        df["SUBSET_EXAMENES"] = df["SUBSET_EXAMENES"]

    return df, source_col


def coerce_binary(series: pd.Series) -> pd.Series:
    """Convierte representaciones frecuentes de variables binarias a 0/1."""
    if series.dtype == bool:
        return series.astype(int)
    s = series.copy()
    if pd.api.types.is_numeric_dtype(s):
        return s.where(s.isna(), (s > 0).astype(int))
    normalized = (
        s.astype(str)
         .str.strip()
         .str.upper()
         .replace({"NAN": np.nan, "NONE": np.nan, "NULL": np.nan, "": np.nan})
    )
    mapping = {
        "1": 1, "SI": 1, "SÍ": 1, "TRUE": 1, "VERDADERO": 1, "POSITIVO": 1, "PRESENTE": 1, "YES": 1,
        "0": 0, "NO": 0, "FALSE": 0, "FALSO": 0, "NEGATIVO": 0, "AUSENTE": 0,
    }
    return normalized.map(mapping)


def dataset_hash(df: pd.DataFrame) -> str:
    """Calcula un hash reproducible del contenido tabular."""
    payload = pd.util.hash_pandas_object(df.fillna("<NA>"), index=True).values.tobytes()
    return hashlib.sha256(payload).hexdigest()


def safe_sheet_name(name: str) -> str:
    return name[:31]


def write_dataset(path: Path, df: pd.DataFrame, variables: pd.DataFrame | None = None) -> None:
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="DATASET", index=False)
        if variables is not None:
            variables.to_excel(writer, sheet_name="VARIABLES", index=False)


## 4. Carga del dataset multimodal integrado

El archivo debe contener la cohorte integrada con identificadores anónimos, variables clínicas, variables NLP y variables ECG.

In [5]:
df = read_excel_prefer_sheet(INPUT_MULTIMODAL, preferred_sheet="BASE_COMPLETA")
print("Dimensiones:", df.shape)
display(df.head())

Archivo cargado: Dataset_Multimodal_Integrado_TFM.xlsx
Hoja utilizada: BASE_MULTIMODAL
Dimensiones: (3779, 101)


,REGISTRO_ID,PACIENTE_ID,UltimaAtencion_Anio,UltimaAtencion_Mes,DiasDesdePrimeraAtencion,Edad,Sexo,Peso,Altura,IMC,PA_Sistolica,PA_Diastolica,Glicemia,ColesterolTotal,HDL,LDL,Trigliceridos,Hemoglobina,Creatinina,Tabaquismo,Diabetes,Peso_kg,Altura_m,IMC_num,PA_Sistolica_mmHg,PA_Diastolica_mmHg,Glicemia_mg_dl,ColesterolTotal_mg_dl,HDL_mg_dl,LDL_mg_dl,Trigliceridos_mg_dl,Hemoglobina_gr_pct,Creatinina_mg_dl,Tabaquismo_bin,Diabetes_bin,ANT_HTA,ANT_DIABETES,ANT_DISLIPIDEMIA,ANT_TABAQUISMO,ANT_OBESIDAD,ANT_INFARTO,ANT_ACV,ANT_CARDIOPATIA,ANT_ARRITMIA,ANT_ENF_RENAL,ANT_EPOC,ANT_CANCER,ANT_HIPOTIROIDISMO,ANT_SALUD_MENTAL,ANT_ALCOHOL,ANT_ASMA,AntecedentesMedicos_Normalizado,AntecedentesMedicos_TextoOriginal_Presente,FLAG_PA_SISTOLICA_ALTA,FLAG_PA_DIASTOLICA_ALTA,FLAG_GLICEMIA_ALTA,FLAG_LDL_ALTO,FLAG_TRIGLICERIDOS_ALTOS,FLAG_OBESIDAD_IMC,FLAG_HDL_BAJO,TOTAL_RESULTADOS,TOTAL_VARIABLES_EVALUADAS,PORCENTAJE_COMPLETITUD,BATERIA_CLUSTER,SUBSET_EXAMENES,SUBSET_BATERIA,PACIENTE_ID_crosswalk,fecha_atencion_norm,estado_validacion,metodo_match,observaciones,ECG_ID,ano,mes,fecha_examen,paciente_id_pdf,sexo,edad,ECG_HR,ECG_PR,ECG_QRS,ECG_QTC,ECG_AXIS,QT,QRS_AXIS,RV5,SV1,RV1,SV5,ECG_ANALYSIS,ECG_DIAGNOSIS,pdf_valido,parametros_extraidos,observaciones_extraccion,fecha,num_ecg_mismo_paciente_fecha,duplicado_mismo_dia,rank_ecg_mismo_paciente_fecha,estado_match_ecg,flag_crosswalk_disponible,flag_ecg_integrado
0,REG_000002,PAC_000002,2026.0,2.0,1227.0,48,Masculino,79 Kg,1.72 mts,26.70,135 mm/Hg,85 mm/Hg,89 mg/dl,244 mg/dl,47 mg/dl,176 mg/dl,258 mg/dl,16.7 gr%,1.0 mg/dl,Sí,No,79.0,1.72,26.70,135.0,85.0,89.0,244.0,47.0,176.0,258.0,16.7,1.00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,1,1,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A,PAC_000002,2026-02-26,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,REG_000007,PAC_000007,2025.0,2.0,847.0,59,Masculino,95 Kg,1.74 mts,31.38,150 mm/Hg,100 mm/Hg,92 mg/dl,231 mg/dl,59 mg/dl,138 mg/dl,201 mg/dl,15 gr%,1.31 mg/dl,Sí,No,95.0,1.74,31.38,150.0,100.0,92.0,231.0,59.0,138.0,201.0,15.0,1.31,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,1,1,0,1,1,1,0,31,31,100.0,2,BATERIA_A,BATERIA_A,PAC_000007,2025-02-11,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,REG_000008,PAC_000008,2025.0,4.0,910.0,28,Masculino,81 Kg,1.71 mts,27.70,138 mm/Hg,70 mm/Hg,100 mg/dl,274 mg/dl,46 mg/dl,131 mg/dl,424 mg/dl,15.4 gr%,1.07 mg/dl,No,No,81.0,1.71,27.70,138.0,70.0,100.0,274.0,46.0,131.0,424.0,15.4,1.07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,1,1,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A,PAC_000008,2025-04-15,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
3,REG_000015,PAC_000015,2025.0,8.0,1030.0,49,Masculino,98 Kg,1.70 mts,33.91,134 mm/Hg,82 mm/Hg,94 mg/dl,191 mg/dl,55 mg/dl,82 mg/dl,178 mg/dl,16 gr%,1.25 mg/dl,No,No,98.0,1.70,33.91,134.0,82.0,94.0,191.0,55.0,82.0,178.0,16.0,1.25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,0,1,1,0,31,31,100.0,2,BATERIA_A,BATERIA_A,PAC_000015,2025-08-13,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
4,REG_000023,PAC_000023,2026.0,3.0,1234.0,38,Masculino,80 Kg,1.83 mts,24.38,133 mm/Hg,74 mm/Hg,99 mg/dl,145 mg/dl,60 mg/dl,93 mg/dl,98 mg/dl,15 gr%,1.1 mg/dl,Sí,No,80.0,1.83,24.38,133.0,74.0,99.0,145.0,60.0,93.0,98.0,15.0,1.10,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,0,0,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A,PAC_000023,2026-03-05,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

## 5. Validación de identificadores y trazabilidad

El pipeline utiliza `PACIENTE_ID` para trazabilidad por paciente y `REGISTRO_ID` para trazabilidad por fila o evento clínico. Si `REGISTRO_ID` no está presente, se genera un identificador técnico de fila para mantener unicidad operativa.

In [6]:
ensure_columns(df, ["PACIENTE_ID"], "dataset multimodal")

if "REGISTRO_ID" not in df.columns:
    warnings.warn("REGISTRO_ID no está presente. Se generará REGISTRO_ID técnico a partir del índice de fila.")
    df.insert(0, "REGISTRO_ID", [f"REG_{i+1:06d}" for i in range(len(df))])

if df["REGISTRO_ID"].duplicated().any():
    duplicated = int(df["REGISTRO_ID"].duplicated().sum())
    raise ValueError(f"REGISTRO_ID debe ser único. Duplicados detectados: {duplicated}")

df, subset_source_col = normalize_bateria_columns(df)
subset_col = "SUBSET_BATERIA"

subset_distribution = (
    df[subset_col]
    .fillna("SIN_SUBSET")
    .value_counts(dropna=False)
    .rename_axis(subset_col)
    .reset_index(name="n")
)
subset_distribution["pct"] = (subset_distribution["n"] / len(df) * 100).round(2)

print("Pacientes únicos:", df["PACIENTE_ID"].nunique(dropna=True))
print("Registros únicos:", df["REGISTRO_ID"].nunique(dropna=True))
print("Columna de subset estándar:", subset_col)
print("Columna fuente de subset:", subset_source_col)
display(subset_distribution)


Pacientes únicos: 3779
Registros únicos: 3779
Columna de subset estándar: SUBSET_BATERIA
Columna fuente de subset: SUBSET_BATERIA


,SUBSET_BATERIA,n,pct
0,BATERIA_B,1192,31.54
1,BATERIA_A,936,24.77
2,BATERIA_C,837,22.15
3,BATERIA_D,814,21.54


## 6. Definición de factores del endpoint

El endpoint operacional se construye a partir de doce factores binarios. Estos factores se utilizan exclusivamente para generar la variable objetivo y se excluyen de los predictores de los datasets E1–E4 para evitar fuga de información directa hacia el modelado.

In [7]:
ENDPOINT_FACTORS = [
    "FLAG_OBESIDAD_IMC",
    "FLAG_PA_SISTOLICA_ALTA",
    "FLAG_PA_DIASTOLICA_ALTA",
    "FLAG_GLICEMIA_ALTA",
    "FLAG_LDL_ALTO",
    "FLAG_TRIGLICERIDOS_ALTOS",
    "FLAG_HDL_BAJO",
    "Diabetes_bin",
    "ANT_HTA",
    "ANT_DIABETES",
    "ANT_DISLIPIDEMIA",
    "ANT_OBESIDAD",
]

ensure_columns(df, ENDPOINT_FACTORS, "factores requeridos para endpoint")

for col in ENDPOINT_FACTORS:
    original_missing = int(df[col].isna().sum())
    df[col] = coerce_binary(df[col])
    invalid = df[col].isna() & df[col].notna()
    if original_missing > 0:
        print(f"{col}: valores faltantes originales = {original_missing}")

endpoint_factor_summary = pd.DataFrame({
    "factor": ENDPOINT_FACTORS,
    "n_no_nulos": [int(df[c].notna().sum()) for c in ENDPOINT_FACTORS],
    "n_faltantes": [int(df[c].isna().sum()) for c in ENDPOINT_FACTORS],
    "n_positivos": [int((df[c] == 1).sum()) for c in ENDPOINT_FACTORS],
    "pct_positivos": [round((df[c] == 1).mean() * 100, 2) for c in ENDPOINT_FACTORS],
})

display(endpoint_factor_summary)

,factor,n_no_nulos,n_faltantes,n_positivos,pct_positivos
0,FLAG_OBESIDAD_IMC,3779,0,1108,29.32
1,FLAG_PA_SISTOLICA_ALTA,3779,0,380,10.06
2,FLAG_PA_DIASTOLICA_ALTA,3779,0,339,8.97
3,FLAG_GLICEMIA_ALTA,3779,0,137,3.63
4,FLAG_LDL_ALTO,3779,0,284,7.52
5,FLAG_TRIGLICERIDOS_ALTOS,3779,0,483,12.78
6,FLAG_HDL_BAJO,3779,0,335,8.86
7,Diabetes_bin,3779,0,98,2.59
8,ANT_HTA,3779,0,20,0.53
9,ANT_DIABETES,3779,0,43,1.14


## 7. Construcción del índice compuesto y endpoint binario

Se calcula:

```text
INDICE_RIESGO_CARDIOMETABOLICO = suma de factores presentes
```

Luego se define:

```text
RIESGO_CARDIOMETABOLICO = 1 si INDICE_RIESGO_CARDIOMETABOLICO >= 3
RIESGO_CARDIOMETABOLICO = 0 si INDICE_RIESGO_CARDIOMETABOLICO < 3
```

Los valores faltantes en factores binarios se tratan como ausencia operacional (`0`) solo para el cómputo del índice, conservando en paralelo el conteo de factores observados y faltantes.

In [8]:
endpoint_matrix = df[ENDPOINT_FACTORS].copy()

df["N_FACTORES_ENDPOINT_OBSERVADOS"] = endpoint_matrix.notna().sum(axis=1)
df["N_FACTORES_ENDPOINT_FALTANTES"] = endpoint_matrix.isna().sum(axis=1)
df["INDICE_RIESGO_CARDIOMETABOLICO"] = endpoint_matrix.fillna(0).sum(axis=1).astype(int)
df["RIESGO_CARDIOMETABOLICO"] = (df["INDICE_RIESGO_CARDIOMETABOLICO"] >= UMBRAL_RIESGO).astype(int)

endpoint_distribution = (
    df["RIESGO_CARDIOMETABOLICO"]
    .value_counts(dropna=False)
    .rename_axis("RIESGO_CARDIOMETABOLICO")
    .reset_index(name="n")
)
endpoint_distribution["pct"] = (endpoint_distribution["n"] / len(df) * 100).round(2)

index_distribution = (
    df["INDICE_RIESGO_CARDIOMETABOLICO"]
    .value_counts()
    .sort_index()
    .rename_axis("INDICE_RIESGO_CARDIOMETABOLICO")
    .reset_index(name="n")
)
index_distribution["pct"] = (index_distribution["n"] / len(df) * 100).round(2)

print("Distribución del endpoint:")
display(endpoint_distribution)
print("Distribución del índice compuesto:")
display(index_distribution)

Distribución del endpoint:


,RIESGO_CARDIOMETABOLICO,n,pct
0,0,3339,88.36
1,1,440,11.64


Distribución del índice compuesto:


,INDICE_RIESGO_CARDIOMETABOLICO,n,pct
0,0,2031,53.74
1,1,879,23.26
2,2,429,11.35
3,3,312,8.26
4,4,88,2.33
5,5,26,0.69
6,6,12,0.32
7,7,2,0.05


## 8. Variables por modalidad

Los predictores se organizan en tres modalidades:

- clínica estructurada;
- NLP clínico;
- ECG estructurado.

Los factores utilizados para construir el endpoint se excluyen de todos los escenarios de modelado.

In [9]:
CLINICAL_CANDIDATES = [
    "Edad", "Sexo",
    "Peso_kg", "Altura_m", "IMC_num",
    "PA_Sistolica_mmHg", "PA_Diastolica_mmHg",
    "Glicemia_mg_dl", "ColesterolTotal_mg_dl", "HDL_mg_dl", "LDL_mg_dl", "Trigliceridos_mg_dl",
    "Hemoglobina_gr_pct", "Creatinina_mg_dl",
    "Tabaquismo_bin",
    "UltimaAtencion_Anio", "UltimaAtencion_Mes", "DiasDesdePrimeraAtencion",
]

NLP_CANDIDATES = [
    "ANT_TABAQUISMO",
    "ANT_INFARTO",
    "ANT_ACV",
    "ANT_CARDIOPATIA",
    "ANT_ARRITMIA",
    "ANT_ENF_RENAL",
    "ANT_EPOC",
    "ANT_CANCER",
    "ANT_HIPOTIROIDISMO",
    "ANT_SALUD_MENTAL",
    "ANT_ALCOHOL",
    "ANT_ASMA",
]

ECG_CANDIDATES = [
    "ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC", "ECG_AXIS",
    "QT", "QRS_AXIS", "RV5", "SV1", "RV1", "SV5",
]

LEAKAGE_COLUMNS = set(ENDPOINT_FACTORS + ["INDICE_RIESGO_CARDIOMETABOLICO"])

clinical_vars = [c for c in existing_columns(df, CLINICAL_CANDIDATES) if c not in LEAKAGE_COLUMNS]
nlp_vars = [c for c in existing_columns(df, NLP_CANDIDATES) if c not in LEAKAGE_COLUMNS]
ecg_vars = [c for c in existing_columns(df, ECG_CANDIDATES) if c not in LEAKAGE_COLUMNS]

if len(clinical_vars) == 0:
    raise ValueError("No se detectaron variables clínicas predictoras disponibles.")
if len(ecg_vars) == 0:
    warnings.warn("No se detectaron variables ECG. Los escenarios E3/E4 no podrán incorporar modalidad ECG.")

print("Variables clínicas:", clinical_vars)
print("Variables NLP:", nlp_vars)
print("Variables ECG:", ecg_vars)
print("Variables excluidas por construcción del endpoint:", sorted(LEAKAGE_COLUMNS))

Variables clínicas: ['Edad', 'Sexo', 'Peso_kg', 'Altura_m', 'IMC_num', 'PA_Sistolica_mmHg', 'PA_Diastolica_mmHg', 'Glicemia_mg_dl', 'ColesterolTotal_mg_dl', 'HDL_mg_dl', 'LDL_mg_dl', 'Trigliceridos_mg_dl', 'Hemoglobina_gr_pct', 'Creatinina_mg_dl', 'Tabaquismo_bin', 'UltimaAtencion_Anio', 'UltimaAtencion_Mes', 'DiasDesdePrimeraAtencion']
Variables NLP: ['ANT_TABAQUISMO', 'ANT_INFARTO', 'ANT_ACV', 'ANT_CARDIOPATIA', 'ANT_ARRITMIA', 'ANT_ENF_RENAL', 'ANT_EPOC', 'ANT_CANCER', 'ANT_HIPOTIROIDISMO', 'ANT_SALUD_MENTAL', 'ANT_ALCOHOL', 'ANT_ASMA']
Variables ECG: ['ECG_HR', 'ECG_PR', 'ECG_QRS', 'ECG_QTC', 'ECG_AXIS', 'QT', 'QRS_AXIS', 'RV5', 'SV1', 'RV1', 'SV5']
Variables excluidas por construcción del endpoint: ['ANT_DIABETES', 'ANT_DISLIPIDEMIA', 'ANT_HTA', 'ANT_OBESIDAD', 'Diabetes_bin', 'FLAG_GLICEMIA_ALTA', 'FLAG_HDL_BAJO', 'FLAG_LDL_ALTO', 'FLAG_OBESIDAD_IMC', 'FLAG_PA_DIASTOLICA_ALTA', 'FLAG_PA_SISTOLICA_ALTA', 'FLAG_TRIGLICERIDOS_ALTOS', 'INDICE_RIESGO_CARDIOMETABOLICO']


## 9. Disponibilidad de ECG

Si no existe `flag_ecg_disponible`, se calcula a partir de la presencia de al menos un parámetro ECG principal.

In [10]:
ECG_MAIN = [c for c in ["ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC", "ECG_AXIS"] if c in df.columns]

if "flag_ecg_disponible" not in df.columns:
    if ECG_MAIN:
        df["flag_ecg_disponible"] = df[ECG_MAIN].notna().any(axis=1).astype(int)
    else:
        df["flag_ecg_disponible"] = 0

coverage_ecg = pd.DataFrame({
    "indicador": ["registros_total", "registros_con_ecg", "porcentaje_con_ecg"],
    "valor": [
        len(df),
        int((df["flag_ecg_disponible"] == 1).sum()),
        round((df["flag_ecg_disponible"] == 1).mean() * 100, 2),
    ],
})
display(coverage_ecg)

,indicador,valor
0,registros_total,3779.00
1,registros_con_ecg,47.00
2,porcentaje_con_ecg,1.24


## 10. Construcción de escenarios experimentales E1–E4

Escenarios definidos:

```text
E1 = Clínico
E2 = Clínico + NLP
E3 = Clínico + ECG
E4 = Clínico + NLP + ECG
```

Todos los escenarios conservan identificadores de trazabilidad, subset estructural, disponibilidad ECG y la variable objetivo. Estos campos de control no deben utilizarse automáticamente como predictores dentro del notebook de modelado.

In [11]:
STRUCTURAL_CONTROL_COLS = [
    "SUBSET_BATERIA",
    "SUBSET_EXAMENES",
    "BATERIA",
    "BATERIA_ESTRUCTURAL",
    "BATERIA_CLUSTER",
    "CLUSTER_BATERIA",
]

CONTROL_COLS = [
    "REGISTRO_ID",
    "PACIENTE_ID",
    *STRUCTURAL_CONTROL_COLS,
    "flag_ecg_disponible",
    "N_FACTORES_ENDPOINT_OBSERVADOS",
    "N_FACTORES_ENDPOINT_FALTANTES",
    "INDICE_RIESGO_CARDIOMETABOLICO",
]
TARGET_COL = "RIESGO_CARDIOMETABOLICO"

CONTROL_COLS = list(dict.fromkeys([c for c in CONTROL_COLS if c in df.columns]))

if "SUBSET_BATERIA" not in CONTROL_COLS:
    raise ValueError("SUBSET_BATERIA debe conservarse como columna de trazabilidad estructural.")

def build_scenario(name: str, feature_cols: list[str]) -> pd.DataFrame:
    cols = CONTROL_COLS + feature_cols + [TARGET_COL]
    cols = list(dict.fromkeys([c for c in cols if c in df.columns]))
    out = df[cols].copy()
    out.insert(0, "ESCENARIO", name)
    return out

scenario_features = {
    "E1_CLINICO": clinical_vars,
    "E2_CLINICO_NLP": clinical_vars + nlp_vars,
    "E3_CLINICO_ECG": clinical_vars + ecg_vars,
    "E4_CLINICO_NLP_ECG": clinical_vars + nlp_vars + ecg_vars,
}

E1 = build_scenario("E1_CLINICO", scenario_features["E1_CLINICO"])
E2 = build_scenario("E2_CLINICO_NLP", scenario_features["E2_CLINICO_NLP"])
E3 = build_scenario("E3_CLINICO_ECG", scenario_features["E3_CLINICO_ECG"])
E4 = build_scenario("E4_CLINICO_NLP_ECG", scenario_features["E4_CLINICO_NLP_ECG"])

for name, dset in [("E1", E1), ("E2", E2), ("E3", E3), ("E4", E4)]:
    print(name, dset.shape, "SUBSET_BATERIA" in dset.columns)


E1 (3779, 29) True
E2 (3779, 41) True
E3 (3779, 40) True
E4 (3779, 52) True


## 11. Diccionario de variables por escenario

In [12]:
variable_rows = []
for scenario, cols in scenario_features.items():
    for col in cols:
        if col in clinical_vars:
            modalidad = "CLINICA"
        elif col in nlp_vars:
            modalidad = "NLP"
        elif col in ecg_vars:
            modalidad = "ECG"
        else:
            modalidad = "OTRA"
        variable_rows.append({
            "escenario": scenario,
            "variable": col,
            "modalidad": modalidad,
            "usada_como_predictor": 1,
            "excluida_por_endpoint": int(col in LEAKAGE_COLUMNS),
        })

for col in ENDPOINT_FACTORS:
    variable_rows.append({
        "escenario": "TODOS",
        "variable": col,
        "modalidad": "FACTOR_ENDPOINT",
        "usada_como_predictor": 0,
        "excluida_por_endpoint": 1,
    })

variables_escenarios = pd.DataFrame(variable_rows)
display(variables_escenarios.head(30))

,escenario,variable,modalidad,usada_como_predictor,excluida_por_endpoint
0,E1_CLINICO,Edad,CLINICA,1,0
1,E1_CLINICO,Sexo,CLINICA,1,0
2,E1_CLINICO,Peso_kg,CLINICA,1,0
3,E1_CLINICO,Altura_m,CLINICA,1,0
4,E1_CLINICO,IMC_num,CLINICA,1,0
5,E1_CLINICO,PA_Sistolica_mmHg,CLINICA,1,0
6,E1_CLINICO,PA_Diastolica_mmHg,CLINICA,1,0
7,E1_CLINICO,Glicemia_mg_dl,CLINICA,1,0
8,E1_CLINICO,ColesterolTotal_mg_dl,CLINICA,1,0
9,E1_CLINICO,HDL_mg_dl,CLINICA,1,0


## 12. Validación de datasets experimentales

In [13]:
def summarize_scenario(name: str, dset: pd.DataFrame, feature_cols: list[str]) -> dict:
    return {
        "escenario": name,
        "n_registros": len(dset),
        "n_columnas": dset.shape[1],
        "n_features": len(feature_cols),
        "n_clinicas": len([c for c in feature_cols if c in clinical_vars]),
        "n_nlp": len([c for c in feature_cols if c in nlp_vars]),
        "n_ecg": len([c for c in feature_cols if c in ecg_vars]),
        "clase_positiva_n": int((dset[TARGET_COL] == 1).sum()),
        "clase_positiva_pct": round((dset[TARGET_COL] == 1).mean() * 100, 2),
        "hash_dataset": dataset_hash(dset),
    }

scenario_summary = pd.DataFrame([
    summarize_scenario("E1_CLINICO", E1, scenario_features["E1_CLINICO"]),
    summarize_scenario("E2_CLINICO_NLP", E2, scenario_features["E2_CLINICO_NLP"]),
    summarize_scenario("E3_CLINICO_ECG", E3, scenario_features["E3_CLINICO_ECG"]),
    summarize_scenario("E4_CLINICO_NLP_ECG", E4, scenario_features["E4_CLINICO_NLP_ECG"]),
])

display(scenario_summary)

,escenario,n_registros,n_columnas,n_features,n_clinicas,n_nlp,n_ecg,clase_positiva_n,clase_positiva_pct,hash_dataset
0,E1_CLINICO,3779,29,18,18,0,0,440,11.64,40175b1b5925f9405438e2dfa3b626354142a92b4f553c...
1,E2_CLINICO_NLP,3779,41,30,18,12,0,440,11.64,4978364169ba6e65b88696437b6981f3f2f698a37e186a...
2,E3_CLINICO_ECG,3779,40,29,18,0,11,440,11.64,18b9074391c96817ab8f07069ca82f6920faa8922ca3e8...
3,E4_CLINICO_NLP_ECG,3779,52,41,18,12,11,440,11.64,900f1fd2d529d4357bb7d1b48b93f00dc379f16b760d94...


## 13. Dataset completo con endpoint

In [14]:
df_endpoint = df.copy()
df_endpoint_hash = dataset_hash(df_endpoint)
print("Dimensiones dataset endpoint:", df_endpoint.shape)
print("Hash dataset endpoint:", df_endpoint_hash)

Dimensiones dataset endpoint: (3779, 106)
Hash dataset endpoint: c306f423b750f9f40af2633fb5c828cd7df64995e843601d3d48cfb4137d9488


## 14. Exportación de resultados

In [ ]:
endpoint_metadata = pd.DataFrame([
    {"campo": "fecha_generacion", "valor": datetime.now().strftime("%Y-%m-%d %H:%M:%S")},
    {"campo": "input_multimodal", "valor": str(INPUT_MULTIMODAL)},
    {"campo": "umbral_endpoint", "valor": UMBRAL_RIESGO},
    {"campo": "target", "valor": TARGET_COL},
    {"campo": "hash_dataset_endpoint", "valor": df_endpoint_hash},
    {"campo": "criterio_leakage", "valor": "factores_endpoint_excluidos_como_predictores"},
    {"campo": "subset_columna_estandar", "valor": "SUBSET_BATERIA"},
    {"campo": "subset_columna_fuente", "valor": subset_source_col},
])

with pd.ExcelWriter(OUTPUT_ENDPOINT, engine="xlsxwriter") as writer:
    df_endpoint.to_excel(writer, sheet_name="DATASET_ENDPOINT", index=False)
    endpoint_distribution.to_excel(writer, sheet_name="DISTRIBUCION_TARGET", index=False)
    index_distribution.to_excel(writer, sheet_name="DISTRIBUCION_INDICE", index=False)
    endpoint_factor_summary.to_excel(writer, sheet_name="FACTORES_ENDPOINT", index=False)
    scenario_summary.to_excel(writer, sheet_name="ESCENARIOS", index=False)
    variables_escenarios.to_excel(writer, sheet_name="VARIABLES_ESCENARIOS", index=False)
    coverage_ecg.to_excel(writer, sheet_name="COBERTURA_ECG", index=False)
    subset_distribution.to_excel(writer, sheet_name="DISTRIBUCION_BATERIA", index=False)
    endpoint_metadata.to_excel(writer, sheet_name="METADATA", index=False)

write_dataset(OUTPUT_E1, E1, variables_escenarios[variables_escenarios["escenario"].isin(["E1_CLINICO", "TODOS"])])
write_dataset(OUTPUT_E2, E2, variables_escenarios[variables_escenarios["escenario"].isin(["E2_CLINICO_NLP", "TODOS"])])
write_dataset(OUTPUT_E3, E3, variables_escenarios[variables_escenarios["escenario"].isin(["E3_CLINICO_ECG", "TODOS"])])
write_dataset(OUTPUT_E4, E4, variables_escenarios[variables_escenarios["escenario"].isin(["E4_CLINICO_NLP_ECG", "TODOS"])])

print("Archivos generados:")
for p in [OUTPUT_ENDPOINT, OUTPUT_E1, OUTPUT_E2, OUTPUT_E3, OUTPUT_E4]:
    print(" -", p)


## 15. Reporte técnico del proceso

In [ ]:
lines = []
lines.append("REPORTE CONSTRUCCION ENDPOINT TFM")
lines.append("=" * 70)
lines.append(f"Fecha generacion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"Entrada principal: {INPUT_MULTIMODAL}")
lines.append(f"Registros procesados: {len(df_endpoint)}")
lines.append(f"Pacientes unicos: {df_endpoint['PACIENTE_ID'].nunique(dropna=True)}")
lines.append(f"Registros unicos: {df_endpoint['REGISTRO_ID'].nunique(dropna=True)}")
lines.append("")
lines.append("TRAZABILIDAD ESTRUCTURAL")
lines.append("-" * 70)
lines.append("Columna estandar conservada en E1-E4: SUBSET_BATERIA")
lines.append(f"Columna fuente utilizada: {subset_source_col}")
lines.append("Distribucion por subconjunto estructural:")
for _, r in subset_distribution.iterrows():
    lines.append(f"  {r['SUBSET_BATERIA']}: {int(r['n'])} registros ({r['pct']}%)")
lines.append("")
lines.append("ENDPOINT")
lines.append("-" * 70)
lines.append(f"Variable objetivo: {TARGET_COL}")
lines.append("Indice: INDICE_RIESGO_CARDIOMETABOLICO")
lines.append(f"Umbral: >= {UMBRAL_RIESGO} factores")
lines.append("Factores utilizados:")
for factor in ENDPOINT_FACTORS:
    lines.append(f"  - {factor}")
lines.append("")
lines.append("Distribucion del endpoint:")
for _, r in endpoint_distribution.iterrows():
    lines.append(f"  Clase {r['RIESGO_CARDIOMETABOLICO']}: {int(r['n'])} registros ({r['pct']}%)")
lines.append("")
lines.append("CONTROL DE LEAKAGE")
lines.append("-" * 70)
lines.append("Los factores usados para construir RIESGO_CARDIOMETABOLICO fueron excluidos como predictores en E1-E4.")
lines.append("SUBSET_BATERIA se conserva como trazabilidad estructural y no debe utilizarse como predictor.")
lines.append("Columnas excluidas:")
for col in sorted(LEAKAGE_COLUMNS):
    lines.append(f"  - {col}")
lines.append("")
lines.append("ESCENARIOS")
lines.append("-" * 70)
for _, r in scenario_summary.iterrows():
    lines.append(
        f"{r['escenario']}: registros={int(r['n_registros'])}, features={int(r['n_features'])}, "
        f"clinicas={int(r['n_clinicas'])}, nlp={int(r['n_nlp'])}, ecg={int(r['n_ecg'])}, "
        f"positivos={int(r['clase_positiva_n'])} ({r['clase_positiva_pct']}%)"
    )
lines.append("")
lines.append("COBERTURA ECG")
lines.append("-" * 70)
for _, r in coverage_ecg.iterrows():
    lines.append(f"{r['indicador']}: {r['valor']}")
lines.append("")
lines.append("ARCHIVOS GENERADOS")
lines.append("-" * 70)
for p in [OUTPUT_ENDPOINT, OUTPUT_E1, OUTPUT_E2, OUTPUT_E3, OUTPUT_E4]:
    lines.append(str(p))
lines.append("")
lines.append("CONSIDERACION METODOLOGICA")
lines.append("-" * 70)
lines.append("RIESGO_CARDIOMETABOLICO es un endpoint operacional, academico y experimental. No corresponde a diagnostico clinico ni escala validada externamente.")

report = "\n".join(lines)
OUTPUT_REPORT.write_text(report, encoding="utf-8")
print(report)
print("\nReporte escrito en:", OUTPUT_REPORT)


## 16. Consideraciones metodológicas finales

1. `RIESGO_CARDIOMETABOLICO` es un endpoint operacional construido para evaluación experimental.
2. Los factores utilizados para construir el endpoint quedan excluidos como predictores para evitar leakage directo.
3. Los datasets E1–E4 preservan identificadores de trazabilidad, pero dichos identificadores no deben ser utilizados como variables predictoras.
4. `SUBSET_BATERIA` se conserva en los datasets E1–E4 como columna de trazabilidad estructural para evaluación por subconjuntos; no debe utilizarse como predictor.
5. La evaluación comparativa de modelos se realiza en `06_modelado_predictivo.ipynb`.
6. La evaluación incremental entre escenarios se realiza en `07_evaluacion_incremental_ecg.ipynb`.
